# 🌊 SSCAP VWB Reconstruction: Auditor's Guide

This notebook provides a complete, independent audit trail for the **Volumetric Water Benefit (VWB)** project. It allows any third party to reconstruct the full sensor dataset directly from decentralized protocols (**Base Blockchain** and **Arweave**), bypassing the project's own servers.

## 📖 How to Audit This Project

To verify the data, an auditor must confirm three things:
1. **Origin**: Was the data signed by the official project wallet?
2. **Immutability**: Is the data stored on a permanent network (Arweave)?
3. **Integrity**: Has the data been double-counted or tampered with?

## 🔍 Manual Verification Links
Before running the code, you can manually inspect the ledger:
*   **Blockchain Ledger (EAS)**: [View all anchors on EASScan](https://base.easscan.org/address/0x44622c7da931c65163f9605AdBADB07ea94C62d4)
*   **Example Raw Data (Arweave)**: [View typical daily bundle JSON](https://gateway.irys.xyz/5qv8ab3KQH94EW5j9Z2c1Yrr3aCtW0PPZGVhVYhGiT0)
*   **Smart Contract**: [VWBToken on Base Mainnet](https://base.blockscout.com/address/0x51b0Df5a5eB30ba7f3620098CCc74A06Bf538ffc)

---

### Step 1: Configuration
We define the official constants of the project. 
* **Attester**: The only wallet authorized to sign data anchors.
* **Schema**: The fixed data structure (string date, string arweaveUrl, uint256 recordCount).

In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime

EAS_GRAPHQL_ID = "https://base.easscan.org/graphql"
ATTESTER = "0x44622c7da931c65163f9605AdBADB07ea94C62d4"
SCHEMA_UID = "0xcb00453bf5b719dca06f811d86dc6cca8dd2c660043cd553c686fbbee522af72"

print(f"Auditing Project Wallet: {ATTESTER}")

### Step 2: Querying the Blockchain Ledger
We use the Ethereum Attestation Service (EAS) GraphQL API to download all "Anchors" registered on the Base blockchain.

In [ ]:
def get_all_anchors():
    query = """
    query Attestations($schema: String!, $attester: String!) {
      attestations(where: {
        schemaId: { equals: $schema },
        attester: { equals: $attester }
      }, orderBy: [{ time: desc }]) {
        id
        time
        decodedDataJson
      }
    }
    """
    res = requests.post(EAS_GRAPHQL_ID, json={'query': query, 'variables': {"schema": SCHEMA_UID, "attester": ATTESTER}})
    return res.json()['data']['attestations']

anchors = get_all_anchors()
print(f"Found {len(anchors)} signed anchors on Base Mainnet.")

### Step 3: Rebuilding the Dataset from Arweave
For each anchor found on the blockchain, we extract the **Arweave URL**. We then download the raw JSON sensor data from Arweave's permanent storage.

**Note on De-duplication:** Some daily periods may have been anchored twice (e.g. during a backfill). We use the unique `filename` (SSCAP Record UUID) to ensure no reading is counted twice.

In [ ]:
all_records = []

for anchor in anchors:
    decoded = json.loads(anchor['decodedDataJson'])
    
    # Robust field lookup (handles both arweaveUrl and historical typos)
    url_item = next((i for i in decoded if i['name'].lower() in ['arweaveurl', 'arwevaeurl']), None)
    if not url_item: continue
    
    url = url_item['value']['value']
    print(f"Fetching bundle: {url}")
    
    res = requests.get(url)
    if res.status_code == 200:
        bundle = res.json()
        for record in bundle:
            # Attach blockchain metadata to every record for auditability
            record['blockchain_anchor_id'] = anchor['id']
            record['arweave_source_url'] = url
            all_records.append(record)

raw_df = pd.DataFrame(all_records)
print(f"\nGathered {len(raw_df)} total anchored readings.")

# CLEANING: deduplicate by 'filename' (UUID)
df = raw_df.drop_duplicates(subset=['filename'])
print(f"Cleaned unique sensor readings: {len(df)}")

### Step 4: Final Audit Export
The data is now reconstructed. Each row contains the original sensor reading PLUS the blockchain UID of where it was anchored.

In [ ]:
display(df.head())

output_file = f"audit_report_{datetime.now().strftime('%Y-%m-%d')}.csv"
df.to_csv(output_file, index=False)
print(f"\nAudit Trail Saved to: {output_file}")